# DPO implicit reward: Colab launcher

Three clicks:

1. **Runtime > Change runtime type > T4 GPU > Save.**
2. Click inside the last-but-one code cell (the one that runs `run_all.sh tierA`), then choose **Runtime > Run before**. This runs every cell above it: the GPU print, Drive mount (allow the pop-up), clone, install, results link, and the GPU check.
3. Read the GPU check output. If it ends with `gpu_check finished`, press the play button on the `run_all.sh tierA` cell.

If the session disconnects, do 1 and 2 again, then press play on the `run_all.sh tierA` cell. Finished runs live on Google Drive and are skipped.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
cd /content
if [ -d dpo-implicit-reward ]; then git -C dpo-implicit-reward pull; else git clone https://github.com/shameershahidd/dpo-implicit-reward.git; fi

In [ ]:
%cd /content/dpo-implicit-reward
# colab's preinstalled copies clash with the pinned torch and are too old for peft
!pip uninstall -y torchaudio torchvision torchao
!pip install -q -r requirements.txt

In [ ]:
%%bash
# results/ lives on Drive so finished runs survive a disconnect
mkdir -p /content/drive/MyDrive/dpo_results
if [ ! -L results ]; then rm -rf results; ln -s /content/drive/MyDrive/dpo_results results; fi
ls -la results

In [ ]:
# first thing to check: the GPU is visible, fp16 training works, our scoring matches the trainer, and the speed
!bash scripts/gpu_check.sh

In [ ]:
# run this only after the GPU check output looks right
!bash scripts/run_all.sh tierA

In [ ]:
# zip the results and figures (without the adapter weights, they stay on Drive) and download
import os
from google.colab import files
os.makedirs('figures', exist_ok=True)
!rm -f /content/results_and_figures.zip
!zip -r -q /content/results_and_figures.zip results figures -x "results/*/*/adapter/*"
files.download('/content/results_and_figures.zip')